In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# 1. Loading data
# Load wildfire data
wildfire = pd.read_csv("../data/raw/NFDB_point.csv", sep = ";")
print("Data loaded successfully")

# Load canada data
canada = gpd.read_file("../data/raw/lpr_000b21a_e.zip")
print("Zip loaded successfully")

Data loaded successfully
Zip loaded successfully


In [4]:
# 2. Reproject canada data to epsg:4326
Source_CRS = "EPSG:3347"
TARGET_CRS = "EPSG:4326"

canada = canada.to_crs(TARGET_CRS)

# Check if the reprojection worked
print(canada.crs)

EPSG:4326


In [ ]:
# 3. Exporting Canada data with epsg 4326
canada.to_file("../data/processed/canada_4326.gpkg", driver = "GPKG")
print("Export canada_4326 successful")

In [ ]:
# 4.1 Data cleaning wildfire data part 1
# Dropping columns 
wildfire_clean = wildfire.drop(["FID",
                        "the_geom",
                        "NFDBFIREID",
                        "NAT_PARK",
                        "FIRENAME",
                        "MONTH",
                        "DAY",
                        "REP_DATE",
                        "OUT_DATE",
                        "FIRE_TYPE",
                        "RESPONSE",
                        "PROTZONE",
                        "MORE_INFO"],
axis = 1) # axis = 1 --> columns


# Renaming columns
new_names = {
    "SRC_AGENCY": "province",
    "FIRE_ID": "fire_id",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "YEAR": "year",
    "CAUSE": "cause",
    "SIZE_HA": "size_ha",
}

wildfire_clean = wildfire_clean.rename(columns=new_names)
wildfire_clean.columns

In [ ]:
# 4.2 Data cleaning wildfire data part 2
# Dropping rows which have H-PB (= Prescribed Burn) as cause
wildfire_clean = wildfire_clean[wildfire_clean['cause'] != 'H-PB']
print((wildfire_clean['cause'] == 'H-PB').sum())

# Dropping row with wrong longitude
fire_clean = wildfire_clean.drop(wildfire_clean[wildfire_clean["longitude"] > 0].index)

print(f"Rows in wildfire_clean:{len(wildfire_clean)}")
print(f"Rows in fire_clean:{len(fire_clean)}")

In [ ]:
# 4.3 Data cleaning wildfire data part 3
# Check if a column has NaNs
print(fire_clean["province"].hasnans)
print(fire_clean["fire_id"].hasnans)
print(fire_clean["size_ha"].hasnans)
print(fire_clean["year"].hasnans)

In [ ]:
# 4.4 Data cleaning wildfire data part 4
# Turning the initials of provinces into full names
province_full = {
    "AB": "Alberta",
    "BC": "British Columbia",
    "MB": "Manitoba",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia",
    "NT": "Northwest Territories",
    "ON": "Ontario",
    "PC": "Parc Canada",
    "QC": "Quebec",
    "SK": "Saskatchewan",
    "YT": "Yukon"
}

fire_clean["province"] = fire_clean["province"].map(province_full).fillna(fire_clean["province"])

print(fire_clean["province"].value_counts())

In [ ]:
# 4.5. Data cleaning wildfire data part 5
# Assigning "Parc Canada" wildfires to the correct provinces

mask = fire_clean["province"] == "Parc Canada" # Only choose rows with "Parc Canada"

# Making GeoDataFrame
geometry = [
    Point(xy) for xy in zip(fire_clean.loc[mask, "longitude"],
                            fire_clean.loc[mask, "latitude"])
]

gdf_points = gpd.GeoDataFrame(
    fire_clean.loc[mask].copy(),
    geometry=geometry,
    crs="EPSG:4326"
)

# IMPORTANT:
# Check name of province column
print(province_borders.columns)

# Spatial join
joined = gpd.sjoin(
    gdf_points,
    province_borders,
    how="left",
    predicate="within"
)

fire_clean.loc[mask, "province"] = joined["PRENAME"].values

print(fire_clean["province"].value_counts())
print((fire_clean["province"] == "Parc Canada").sum())

In [ ]:
# 4.6 Data cleaning wildfire data part 6
# Turning the initials of cause into full names
cause_name = {
    "H": "Human",
    "N": "Natural",
    "U": "Unknown"
}

fire_clean["cause"] = fire_clean["cause"].map(cause_name).fillna(fire_clean["cause"])

In [ ]:
# 5. Exporting cleand fire data
fire_clean.to_csv("../data/processed/fire_clean.csv", index = False)
print("Export successful")

In [ ]:
# 6. Only keeping the years 2014 - 2023

# To know which years are available
print(fire_clean["year"].unique())

# Dropping row with years I don't need
fire_14_23 = fire_clean.drop(fire_clean[fire_clean["year"] <2014].index)

# To know if the dropping worked
print(fire_14_23["year"].unique())

In [ ]:
# 7. Exporting fire data for 2014-2023
fire_14_23.to_csv("../data/processed/fire_14_23.csv", index = False)
print("Export successful")